# AutoMixAI – Genre Classifier Training

Trains a Deep Neural Network (DNN) on the **GTZAN Music Genre Classification** dataset to predict one of 10 genres:
> `blues · classical · country · disco · hiphop · jazz · metal · pop · reggae · rock`

### Dataset
Kaggle: [andradaolteanu/gtzan-dataset-music-genre-classification](https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification)

This notebook uses the **pre-computed 57-feature CSV** (`features_3_sec.csv`) which gives ~10,000 samples (100 songs × 10 genres × 10 segments).

### Output
| File | Description |
|------|-------------|
| `genre_classifier.h5` | Keras model (input: 57 features, output: 10 class softmax) |
| `genre_scaler.pkl` | sklearn `StandardScaler` fitted on training features |

Copy both to `backend/app/storage/models/` to enable model-based genre classification in AutoMixAI.

In [ ]:
# Install / confirm packages (all pre-installed on Kaggle)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tensorflow', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn'], check=True)
print('All packages ready.')

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

print(f'TF version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Locate dataset ────────────────────────────────────────────────────
# Kaggle mounts the dataset at /kaggle/input/gtzan-dataset-music-genre-classification/
DATASET_ROOT = '/kaggle/input/gtzan-dataset-music-genre-classification'

csv_3sec  = os.path.join(DATASET_ROOT, 'Data', 'features_3_sec.csv')
csv_30sec = os.path.join(DATASET_ROOT, 'Data', 'features_30_sec.csv')

# Prefer 3-sec CSV (10× more rows → better generalisation)
csv_path = csv_3sec if os.path.exists(csv_3sec) else csv_30sec
print(f'Using: {csv_path}')

df = pd.read_csv(csv_path)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
print('Columns:', df.columns.tolist())
print('\nClass distribution:')
print(df['label'].value_counts())

In [ ]:
# ── Select the 57 features that AutoMixAI computes at inference time ─
#
# Layout expected by genre_classifier.py:
#   [0]  chroma_stft_mean
#   [1]  chroma_stft_var
#   [2]  rms_mean
#   [3]  rms_var
#   [4]  spectral_centroid_mean
#   [5]  spectral_centroid_var
#   [6]  spectral_bandwidth_mean
#   [7]  spectral_bandwidth_var
#   [8]  rolloff_mean
#   [9]  rolloff_var
#  [10]  zero_crossing_rate_mean
#  [11]  zero_crossing_rate_var
#  [12]  harmony_mean     (harmonic RMS via HPSS)
#  [13]  harmony_var
#  [14]  perceptr_mean    (percussive RMS via HPSS)
#  [15]  perceptr_var
#  [16]  tempo
#  [17-36]  mfcc1_mean .. mfcc20_mean
#  [37-56]  mfcc1_var  .. mfcc20_var

# Map GTZAN CSV column names → our 57-slot order
FEATURE_COLS = [
    'chroma_stft_mean', 'chroma_stft_var',
    'rms_mean', 'rms_var',
    'spectral_centroid_mean', 'spectral_centroid_var',
    'spectral_bandwidth_mean', 'spectral_bandwidth_var',
    'rolloff_mean', 'rolloff_var',
    'zero_crossing_rate_mean', 'zero_crossing_rate_var',
    'harmony_mean', 'harmony_var',
    'perceptr_mean', 'perceptr_var',
    'tempo',
    'mfcc1_mean',  'mfcc2_mean',  'mfcc3_mean',  'mfcc4_mean',
    'mfcc5_mean',  'mfcc6_mean',  'mfcc7_mean',  'mfcc8_mean',
    'mfcc9_mean',  'mfcc10_mean', 'mfcc11_mean', 'mfcc12_mean',
    'mfcc13_mean', 'mfcc14_mean', 'mfcc15_mean', 'mfcc16_mean',
    'mfcc17_mean', 'mfcc18_mean', 'mfcc19_mean', 'mfcc20_mean',
    'mfcc1_var',   'mfcc2_var',   'mfcc3_var',   'mfcc4_var',
    'mfcc5_var',   'mfcc6_var',   'mfcc7_var',   'mfcc8_var',
    'mfcc9_var',   'mfcc10_var',  'mfcc11_var',  'mfcc12_var',
    'mfcc13_var',  'mfcc14_var',  'mfcc15_var',  'mfcc16_var',
    'mfcc17_var',  'mfcc18_var',  'mfcc19_var',  'mfcc20_var',
]

# Verify all columns exist (3-sec CSV uses different MFC column naming)
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print('Missing columns — checking alternate names:', missing)
    # The 3-sec CSV may use 'mfcc1' prefix without underscore etc.
    # Print all available columns for inspection
    print([c for c in df.columns if 'mfcc' in c.lower()][:10])
else:
    print(f'All {len(FEATURE_COLS)} feature columns found.')

In [ ]:
# ── Auto-fix column name mismatches ──────────────────────────────────
available = set(df.columns)

# Some versions of the CSV use e.g. 'mfcc1' instead of 'mfcc1_mean'
# Build a flexible mapping
def find_col(name):
    if name in available:
        return name
    # Try without _mean/_var suffix variants
    base = name.replace('_mean', '').replace('_var', '')
    suffix = '_mean' if '_mean' in name else ('_var' if '_var' in name else '')
    candidates = [c for c in available if c.startswith(base)]
    for c in candidates:
        if suffix and suffix in c:
            return c
    return candidates[0] if candidates else None

resolved = {c: find_col(c) for c in FEATURE_COLS}
still_missing = [k for k, v in resolved.items() if v is None]
if still_missing:
    print('WARNING — could not resolve these columns:', still_missing)
    print('Available columns:', list(available))
else:
    print('All columns resolved.')

# Build X using resolved column names
X = df[[resolved[c] for c in FEATURE_COLS]].values.astype(np.float32)
print(f'Feature matrix shape: {X.shape}')

In [ ]:
# ── Encode labels ─────────────────────────────────────────────────────
GENRE_ORDER = ['blues', 'classical', 'country', 'disco',
               'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

le = LabelEncoder()
le.classes_ = np.array(GENRE_ORDER)
y = le.transform(df['label'].str.lower())

print('Classes:', le.classes_)
print('y distribution:', np.bincount(y))

In [ ]:
# ── Train / val / test split ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.12, random_state=42, stratify=y_train)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

# ── Normalise features ────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

In [ ]:
# ── Build DNN ─────────────────────────────────────────────────────────
INPUT_DIM  = X_train_s.shape[1]   # 57
N_CLASSES  = len(GENRE_ORDER)      # 10

def build_genre_model(input_dim: int, n_classes: int) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name='features')
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64,  activation='relu')(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='genre')(x)
    model = keras.Model(inputs, outputs, name='genre_classifier')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_genre_model(INPUT_DIM, N_CLASSES)
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────
cbs = [
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15,
        restore_best_weights=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=6,
        min_lr=1e-5, verbose=1,
    ),
]

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=120,
    batch_size=64,
    callbacks=cbs,
    verbose=1,
)
print('Training done.')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle('Genre Classifier Training', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}')

y_pred = np.argmax(model.predict(X_test_s, verbose=0), axis=1)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=GENRE_ORDER))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=GENRE_ORDER, yticklabels=GENRE_ORDER,
)
plt.title('Genre Classifier — Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save model + scaler ───────────────────────────────────────────────
MODEL_PATH  = 'genre_classifier.h5'
SCALER_PATH = 'genre_scaler.pkl'

model.save(MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

model_kb  = os.path.getsize(MODEL_PATH) / 1024
scaler_kb = os.path.getsize(SCALER_PATH) / 1024

print(f'Saved model  → {MODEL_PATH}  ({model_kb:.1f} KB)')
print(f'Saved scaler → {SCALER_PATH}  ({scaler_kb:.1f} KB)')
print()
print('Next step:')
print('  Copy both files to backend/app/storage/models/')
print('  AutoMixAI will auto-load them on next startup.')

In [ ]:
# ── (Optional) Feature importance via permutation ─────────────────────
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

class KerasWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model
        self.classes_ = np.arange(N_CLASSES)
    def fit(self, X, y): return self
    def predict(self, X): return np.argmax(self.model.predict(X, verbose=0), axis=1)
    def score(self, X, y): return np.mean(self.predict(X) == y)

wrapped = KerasWrapper(model)
result_fi = permutation_importance(
    wrapped, X_test_s, y_test,
    n_repeats=5, random_state=42, n_jobs=-1,
)

fi_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': result_fi.importances_mean,
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
plt.barh(fi_df['feature'].head(20)[::-1], fi_df['importance'].head(20)[::-1],
         color='#7c3aed', alpha=0.85)
plt.xlabel('Mean accuracy decrease')
plt.title('Top 20 Most Important Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print(fi_df.head(10).to_string(index=False))

## Deployment Instructions

1. Download `genre_classifier.h5` and `genre_scaler.pkl` from Kaggle output.
2. Copy both to `backend/app/storage/models/`.
3. Restart the AutoMixAI server (`uvicorn app.main:app --reload`).
4. The genre classifier will be auto-loaded on startup.

No config changes needed — the model path is already set in `config.py`:
```
genre_model_filename  = 'genre_classifier.h5'
genre_scaler_filename = 'genre_scaler.pkl'
```

## Feature layout (must match `genre_classifier.py`)

| Index | Feature |
|-------|---------|
| 0 | chroma_stft_mean |
| 1 | chroma_stft_var |
| 2-3 | rms mean/var |
| 4-5 | spectral_centroid mean/var |
| 6-7 | spectral_bandwidth mean/var |
| 8-9 | rolloff mean/var |
| 10-11 | zero_crossing_rate mean/var |
| 12-13 | harmony (HPSS) mean/var |
| 14-15 | perceptr (HPSS) mean/var |
| 16 | tempo |
| 17-36 | mfcc1..mfcc20 mean |
| 37-56 | mfcc1..mfcc20 var |